# 04 — Run and inspect SageMaker pipelines

Start **training or inference** from a published pipeline name/ARN, or use the
publication saved by notebook 02. Model code runs in SageMaker jobs; this
notebook submits requests and reads their status.

To start: run setup → choose inputs → inspect → prepare request → submit.
To check progress: rerun **Monitor an execution**, then inspect outputs.
To inspect an existing run without submitting: run setup, paste its ARN in the
monitor cell, and skip all submission cells.

Use a new `RUN_LABEL` for each intended execution. Reusing the same label for
the same request resumes its saved receipt, including after a kernel restart.
Credentials come from the notebook's normal AWS identity; no keys are entered.

## Read local configuration

- Locate the checkout and read config.json when present.
- Load notebook 02's publication record for the configured namespace.
- Import the AWS client and display tools; make no AWS request yet.

In [ ]:
from pathlib import Path
from uuid import uuid4
from copy import deepcopy
import hashlib
import json
import re
import boto3
import pandas as pd
from botocore.config import Config
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
config_path = ROOT / "notebooks/config.json"
notebook_config = json.loads(config_path.read_text()) if config_path.is_file() else {}
namespace = notebook_config.get("namespace", "").strip()
publication_path = ROOT / ".runs/source-deploy" / namespace / "publication.json"
publication = json.loads(publication_path.read_text()) if namespace and publication_path.is_file() else {}
client_options = Config(connect_timeout=10, read_timeout=60,
                        retries={"mode": "standard", "total_max_attempts": 3})
print("Configured namespace:", namespace or "(none)")
print("Saved publication:", publication_path if publication else "(none; paste a pipeline name or ARN)")

## Choose the pipeline and this execution

- Paste a pipeline name or ARN, or leave it blank to use notebook 02's saved pipeline.
- Choose training/inference for the saved-publication route and a label for this intended run.
- Set only the parameter overrides you need; the next cell displays published defaults.

In [ ]:
PIPELINE = ""                  # Paste a training OR inference pipeline name/ARN.
PIPELINE_KIND = "training"      # Used only when PIPELINE is blank.
PIPELINE_VERSION = None         # None uses the saved version, or latest for a pasted pipeline.
AWS_REGION = ""                # Blank uses the ARN, saved publication, or current AWS region.
RUN_LABEL = "trial-001"         # Change for each new intended run; keep for retries.
PARAMETER_OVERRIDES = {
    # "ModelId": "usecase1",
    # "CsvInputUri": "s3://your-bucket/wsf-mlops/your-namespace/input.csv",
    # Training: "TrainingMode": "Preliminary models with validation",
    # Inference: "ModelPackageArn": "arn:aws:sagemaker:REGION:ACCOUNT:model-package/GROUP/VERSION",
    # Inference: "ModelCodeSource": "artifact",  # "pipeline" tests newly published code.
}

## Inspect the selected version and its parameters

- Resolve a concrete pipeline version and read that version's definition.
- Show the selected account/Region, deployment namespace and effective parameters.
- Reject unknown/missing parameters before preparing a submission.

In [ ]:
inspected_inputs = None
selected_publication = publication.get("pipelines", {}).get(PIPELINE_KIND, {}) if not PIPELINE else {}
pipeline_target = PIPELINE.strip() or selected_publication.get("arn") or selected_publication.get("name")
if not pipeline_target:
    raise ValueError("Paste PIPELINE or run notebook 02 to save a publication for this namespace.")
if "/execution/" in pipeline_target:
    raise ValueError("That is an execution ARN. Paste it in the monitor cell instead.")
arn_region = pipeline_target.split(":")[3] if pipeline_target.startswith("arn:") else ""
region = AWS_REGION.strip() or arn_region or publication.get("region") or None
if arn_region and region != arn_region:
    raise ValueError("AWS_REGION differs from the pipeline ARN's Region.")
session = boto3.Session(region_name=region)
if not session.region_name:
    raise ValueError("Set AWS_REGION when using a pipeline name without a configured Region.")
sm = session.client("sagemaker", config=client_options)
if not hasattr(sm, "list_pipeline_versions"):
    raise RuntimeError(f"Boto3 {boto3.__version__} lacks pipeline-version APIs. Use the deployment notebook's supported AWS SDK environment.")
pipeline_version = PIPELINE_VERSION or selected_publication.get("version")
if pipeline_version is None:
    versions = sm.list_pipeline_versions(PipelineName=pipeline_target, SortOrder="Descending", MaxResults=1)
    if not versions["PipelineVersionSummaries"]:
        raise ValueError("The selected pipeline has no published version.")
    pipeline_version = versions["PipelineVersionSummaries"][0]["PipelineVersionId"]
description = sm.describe_pipeline(PipelineName=pipeline_target, PipelineVersionId=int(pipeline_version))
pipeline_arn = description["PipelineArn"]
definition = json.loads(description["PipelineDefinition"])
declared = {p["Name"]: p for p in definition.get("Parameters", [])}
unknown = set(PARAMETER_OVERRIDES) - set(declared)
if unknown:
    raise ValueError(f"Unknown parameters for this pipeline: {sorted(unknown)}")
effective = {name: p["DefaultValue"] for name, p in declared.items() if "DefaultValue" in p}
effective.update(PARAMETER_OVERRIDES)
missing = set(declared) - set(effective)
if missing:
    raise ValueError(f"Supply these parameters in PARAMETER_OVERRIDES: {sorted(missing)}")
for name, parameter in declared.items():
    if "EnumValues" in parameter and effective[name] not in parameter["EnumValues"]:
        raise ValueError(f"{name} must be one of {parameter['EnumValues']}")
inspected_inputs = deepcopy((PIPELINE, PIPELINE_KIND, PIPELINE_VERSION, AWS_REGION,
                              RUN_LABEL, PARAMETER_OVERRIDES))
print("Pipeline:", pipeline_arn, "| version:", pipeline_version)
print("Region:", session.region_name, "| namespace:", definition.get("Metadata", {}).get("DeploymentNamespace", "not recorded"))
display(pd.DataFrame([{"parameter": name, "value": value,
                       "source": "override" if name in PARAMETER_OVERRIDES else "published default"}
                      for name, value in effective.items()]))

## Inspect the inference model when applicable

- Keep latest-approved to resolve the model during execution, or supply an exact package ARN.
- When using an exact version, display its approval, namespace, model ID and artifact location.
- Leave Registry eligibility checks in the pipeline, where failures explain the specific mismatch.

This cell does not approve or modify a model. Notebook 02 has the optional
exact-version approval cell. Keep `ModelMetadataUri=registry` for the ordinary
inference path. Training uses its own published metadata/mode defaults.

In [ ]:
if "ModelPackageArn" in declared:
    model_package_arn = str(effective["ModelPackageArn"])
    if model_package_arn == "latest-approved":
        print("The pipeline will resolve its newest eligible approved model during execution.")
        print("Use an exact ModelPackageArn when repeating a particular model version.")
    elif model_package_arn.startswith("arn:"):
        package = sm.describe_model_package(ModelPackageName=model_package_arn)
        properties = package.get("CustomerMetadataProperties", {})
        display({"package": package["ModelPackageArn"],
                 "approval": package.get("ModelApprovalStatus"),
                 "lifecycle": package.get("ModelLifeCycle", {}),
                 "namespace": properties.get("DeploymentNamespace"), "model": properties.get("ModelId"),
                 "containers": package.get("InferenceSpecification", {}).get("Containers", [])})
    else:
        raise ValueError("ModelPackageArn must be 'latest-approved' or an exact package ARN.")
else:
    print("Training: no existing model approval is needed. Review the newly registered version after training.")

## Save the intended request

- Build the version-pinned request with your explicit parameter overrides.
- Save a request token before submission; the same label reuses the same request.
- Keep receipts separate for each pipeline, so different namespaces can run concurrently.

In [ ]:
if inspected_inputs != (PIPELINE, PIPELINE_KIND, PIPELINE_VERSION, AWS_REGION, RUN_LABEL, PARAMETER_OVERRIDES):
    raise ValueError("Inputs changed. Rerun Inspect first.")
if len(RUN_LABEL) > 82 or not re.fullmatch(r"[A-Za-z0-9](?:-*[A-Za-z0-9]){0,81}", RUN_LABEL):
    raise ValueError("RUN_LABEL must contain 1–82 letters/digits/hyphens and start/end with a letter or digit.")
request = {
    "PipelineName": pipeline_arn,
    "PipelineVersionId": int(pipeline_version),
    "PipelineExecutionDisplayName": RUN_LABEL,
    "PipelineParameters": [{"Name": name, "Value": str(value).lower() if isinstance(value, bool) else str(value)}
                           for name, value in sorted(PARAMETER_OVERRIDES.items())],
}
receipt_directory = ROOT / ".runs/pipeline-executions" / hashlib.sha256(pipeline_arn.encode()).hexdigest()[:16]
receipt_directory.mkdir(parents=True, exist_ok=True)
receipt_path = receipt_directory / (RUN_LABEL + ".json")
receipt = {"request": request, "client_request_token": uuid4().hex, "region": session.region_name,
           "effective_parameters": effective}
try:
    with receipt_path.open("x") as stream:
        json.dump(receipt, stream, indent=2)
except FileExistsError:
    receipt = json.loads(receipt_path.read_text())
    if receipt["request"] != request:
        raise ValueError("This RUN_LABEL already identifies a different request. Choose a new RUN_LABEL.")
print("Request saved:", receipt_path)
display(receipt["request"])

## Submit this execution

- Start the reviewed request, or reuse the execution already saved in its receipt.
- Reuse the saved request token after a network interruption rather than starting a duplicate.
- Save and print the execution ARN; the jobs continue after this notebook closes.

**Effect:** this cell can start billable SageMaker jobs. Inspect the pipeline,
version and parameters printed above before running it. To start another run,
change `RUN_LABEL` and repeat the selection/request cells.

In [ ]:
if inspected_inputs != (PIPELINE, PIPELINE_KIND, PIPELINE_VERSION, AWS_REGION, RUN_LABEL, PARAMETER_OVERRIDES):
    raise ValueError("Inputs changed. Rerun Inspect and Save request before submitting.")
receipt = json.loads(receipt_path.read_text())
if receipt["request"] != request:
    raise ValueError("The saved request changed. Inspect the receipt before submitting.")
if receipt.get("execution_arn"):
    execution_arn = receipt["execution_arn"]
    print("Using the saved execution; no new start request sent.")
else:
    response = sm.start_pipeline_execution(**receipt["request"], ClientRequestToken=receipt["client_request_token"])
    execution_arn = response["PipelineExecutionArn"]
    receipt["execution_arn"] = execution_arn
    temporary_receipt = receipt_path.with_suffix("." + uuid4().hex + ".tmp")
    temporary_receipt.write_text(json.dumps(receipt, indent=2) + "\n")
    temporary_receipt.replace(receipt_path)
print("Execution:", execution_arn)
print("Receipt:", receipt_path)

## Monitor an execution — safe to rerun

- Paste an existing execution ARN, or inspect the execution submitted above.
- Read status and every page of step results; show failures and cache-hit producers.
- Read the execution's actual parameters without starting or modifying jobs.

Refresh this cell while several namespaces execute in parallel. The selected
execution ARN determines which run is inspected; changing it does not submit
another execution. After restarting the kernel, run setup and paste the ARN.

In [ ]:
MONITOR_EXECUTION_ARN = ""  # Paste here to inspect any existing training/inference execution.
selected_execution = MONITOR_EXECUTION_ARN.strip() or globals().get("execution_arn", "")
if not selected_execution or "/execution/" not in selected_execution:
    raise ValueError("Paste an execution ARN here, or submit a request above.")
monitor_session = boto3.Session(region_name=selected_execution.split(":")[3])
monitor_sm = monitor_session.client("sagemaker", config=client_options)
execution = monitor_sm.describe_pipeline_execution(PipelineExecutionArn=selected_execution)
steps = []
for page in monitor_sm.get_paginator("list_pipeline_execution_steps").paginate(PipelineExecutionArn=selected_execution):
    steps.extend(page["PipelineExecutionSteps"])
step_rows = []
for step in steps:
    metadata = step.get("Metadata", {})
    jobs = {kind: details["Arn"] for kind, details in metadata.items()
            if isinstance(details, dict) and "Arn" in details}
    step_rows.append({"step": step["StepName"], "status": step["StepStatus"],
                      "failure": step.get("FailureReason", ""),
                      "cached_from": step.get("CacheHitResult", {}).get("SourcePipelineExecutionArn", ""),
                      "resources": jobs})
print("Execution:", selected_execution)
print("Status:", execution["PipelineExecutionStatus"], "| version:", execution.get("PipelineVersionId"))
if execution.get("FailureReason"):
    print("Failure:", execution["FailureReason"])
display(pd.DataFrame(step_rows))
actual_parameters = []
for page in monitor_sm.get_paginator("list_pipeline_parameters_for_execution").paginate(PipelineExecutionArn=selected_execution):
    actual_parameters.extend(page["PipelineParameters"])
display(pd.DataFrame(actual_parameters))

## Find outputs from the producing jobs

- Describe the jobs linked by the selected execution's steps.
- Show their actual model/report/output S3 locations, including cached producers.
- Retain Registry version ARNs and job failure details for review.

In [ ]:
artifact_rows = []
for step in steps:
    for kind, item in step.get("Metadata", {}).items():
        arn = item.get("Arn") if isinstance(item, dict) else None
        if not arn:
            continue
        name = arn.rsplit("/", 1)[-1]
        if kind == "ProcessingJob":
            job = monitor_sm.describe_processing_job(ProcessingJobName=name)
            for output in job.get("ProcessingOutputConfig", {}).get("Outputs", []):
                artifact_rows.append({"step": step["StepName"], "artifact": output["OutputName"],
                                      "uri": output.get("S3Output", {}).get("S3Uri", ""), "producer": arn})
        elif kind == "TrainingJob":
            job = monitor_sm.describe_training_job(TrainingJobName=name)
            artifact_rows.append({"step": step["StepName"], "artifact": "model",
                                  "uri": job.get("ModelArtifacts", {}).get("S3ModelArtifacts", ""), "producer": arn})
            artifact_rows.append({"step": step["StepName"], "artifact": "training output prefix",
                                  "uri": job.get("OutputDataConfig", {}).get("S3OutputPath", ""), "producer": arn})
        elif kind == "TransformJob":
            job = monitor_sm.describe_transform_job(TransformJobName=name)
            artifact_rows.append({"step": step["StepName"], "artifact": "transform",
                                  "uri": job.get("TransformOutput", {}).get("S3OutputPath", ""), "producer": arn})
        elif kind == "RegisterModel":
            artifact_rows.append({"step": step["StepName"], "artifact": "model package",
                                  "uri": arn, "producer": arn})
display(pd.DataFrame(artifact_rows))

## Read a selected report or registration receipt

- Optionally paste an exact JSON object URI from an output location above.
- Read that object with the notebook identity and inspect its contents.
- Use recorded model-package and tracking references for the next experiment.

Pipeline status comes from SageMaker. MLflow metrics/artifacts and final run
status depend on the configured tracking and completion services; a successful
pipeline alone does not establish that tracking completed. The custom Registry
step writes its package ARN into its registration output.

In [ ]:
JSON_ARTIFACT_URI = ""  # e.g. the registry output prefix plus /package.json, or /run.json.
if JSON_ARTIFACT_URI:
    from urllib.parse import urlsplit
    location = urlsplit(JSON_ARTIFACT_URI)
    if location.scheme != "s3" or not location.netloc or not location.path.strip("/"):
        raise ValueError("Use the complete s3://bucket/key.json URI.")
    response = monitor_session.client("s3", config=client_options).get_object(
        Bucket=location.netloc, Key=location.path.lstrip("/"))
    report = json.loads(response["Body"].read())
    display(report)
else:
    print("No object selected. The output locations remain available in artifact_rows.")